# OpenAI Embeddings: SDK vs REST API

This notebook demonstrates how to use both the OpenAI SDK-based and REST API-based embeddings LLMs.

In [1]:
# Install required dependencies
# %pip install openai httpx pydantic
from pathlib import Path
import os
import sys

root_path = Path("__file__").absolute().parent.parent
sys.path.append(os.path.join(root_path))
# print(os.path.join(root_path, "fnllm"))

In [2]:
import asyncio
import os
from fnllm.openai.factories import create_openai_embeddings_llm, create_openai_embeddings_rest_llm
from fnllm.openai.config import PublicOpenAIConfig, AzureOpenAIConfig

## Configuration

Set up your API configuration. You'll need your OpenAI API key.

In [3]:
# Configuration
API_KEY = os.getenv("OPENAI_API_KEY", "your-api-key-here")
BASE_URL = "https://azuresqlcopilotdev-validation-openai.openai.azure.com/"  # For OpenAI
MODEL = "text-embedding-3-large"

from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(exclude_environment_credential=True, exclude_shared_token_cache_credential=True),
    "https://cognitiveservices.azure.com/.default"
)

# For Azure OpenAI, you would use:
# BASE_URL = "https://your-resource.openai.azure.com"
# API_VERSION = "2024-02-01"
# MODEL = "your-deployment-name"

# Sample texts to embed
texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Python is a powerful programming language.",
    "Machine learning is transforming industries."
]

## Method 1: Using OpenAI SDK (Original Implementation)

In [4]:
async def test_sdk_embeddings():
    """Test embeddings using the OpenAI SDK."""

    # Create config for SDK-based LLM
    # config = PublicOpenAIConfig(
    #     api_key=API_KEY,
    #     base_url=BASE_URL,
    #     model=MODEL
    # )
    config = AzureOpenAIConfig(
        endpoint=BASE_URL,
        deployment=MODEL,
        api_version="2024-02-01"
    )

    # Create the SDK-based embeddings LLM
    llm = create_openai_embeddings_llm(config)

    print("=== Using OpenAI SDK ===")

    for i, text in enumerate(texts):
        result = await llm(text)
        print(f"Text {i+1}: {text}")
        print(f"Embedding dimensions: {len(result.output.embeddings[0])}")
        print(f"First 5 values: {result.output.embeddings[0][:5]}")
        print(f"Usage: {result.output.usage}")
        print("---")

    return llm, result

# Run the SDK test
sdk_llm, sdk_result = await test_sdk_embeddings()

=== Using OpenAI SDK ===
Text 1: The quick brown fox jumps over the lazy dog.
Embedding dimensions: 3072
First 5 values: [-0.012696055695414543, 0.009628890082240105, -0.011472541838884354, 0.0206488985568285, 0.0013094117166474462]
Usage: input_tokens=10 output_tokens=0 total_tokens=10
---
Text 2: Python is a powerful programming language.
Embedding dimensions: 3072
First 5 values: [-0.00022397100110538304, -0.013018861413002014, -0.006049690768122673, 0.04973716661334038, 0.008792907930910587]
Usage: input_tokens=7 output_tokens=0 total_tokens=7
---
Text 3: Machine learning is transforming industries.
Embedding dimensions: 3072
First 5 values: [0.0199761800467968, 0.02161978930234909, -0.014890823513269424, -0.03146740049123764, -0.015466788783669472]
Usage: input_tokens=6 output_tokens=0 total_tokens=6
---


In [5]:
sdk_result.output.headers

Headers({'content-length': '16606', 'content-type': 'application/json', 'access-control-allow-origin': '*', 'apim-request-id': '515e5c3d-0ea3-41f6-80ee-0dd8a089d0b3', 'strict-transport-security': 'max-age=31536000; includeSubDomains; preload', 'x-content-type-options': 'nosniff', 'x-ms-region': 'Canada East', 'x-ratelimit-remaining-tokens': '350967', 'x-ratelimit-limit-tokens': '351000', 'x-request-id': '03e38f18-16a0-4985-bee0-fd5956edd735', 'azureml-model-session': 'd041-20250522153657', 'x-envoy-upstream-service-time': '38', 'x-ms-client-request-id': 'Not-Set', 'x-ms-deployment-name': 'text-embedding-3-large', 'date': 'Fri, 29 Aug 2025 17:52:39 GMT'})

In [6]:
sdk_result.output.raw_model

CreateEmbeddingResponse(data=[Embedding(embedding=[0.0199761800467968, 0.02161978930234909, -0.014890823513269424, -0.03146740049123764, -0.015466788783669472, -0.018416857346892357, -0.015059398487210274, 0.03559749573469162, 0.011420980095863342, 0.005264468025416136, 0.006978317629545927, -0.03795755282044411, 0.0009859902784228325, -0.0007590281311422586, -0.026199420914053917, -0.006922125816345215, 0.022912200540304184, -0.01979355700314045, -0.034305084496736526, -0.03585036098957062, -0.011140021495521069, -0.03315315395593643, -0.02975355088710785, 0.023151015862822533, -0.026775386184453964, 0.013331500813364983, 0.012608031742274761, -0.0043232557363808155, -0.036777522414922714, 0.02164788544178009, 0.01980760507285595, 0.001708581461571157, 0.04790349677205086, -0.014497480355203152, -0.02176026999950409, 0.024401282891631126, -0.02146526239812374, 0.025005344301462173, 0.007663154974579811, -0.006901053711771965, 0.04739776998758316, -0.008414720185101032, 0.0054119713604

In [7]:
sdk_result.output.raw_model.object

'list'

In [8]:
type(sdk_result.output.raw_model.usage)

openai.types.create_embedding_response.Usage

In [9]:
type(sdk_result.output.raw_model.data[0])

openai.types.embedding.Embedding

In [10]:
from openai.types.embedding import Embedding

## Method 2: Using REST API (New Implementation)

In [13]:
async def test_rest_embeddings():
    """Test embeddings using REST API calls."""
    BASE_URL = "https://data-ai-dev.microsoft.com"
    API_KEY = "384ea53d-9011-5e6e-35ab-87d3b24abfde"
    MODEL = "metisaca"
    # Create the REST-based embeddings LLM
    llm = create_openai_embeddings_rest_llm(
        base_url=BASE_URL,
        api_key=API_KEY,
        model=MODEL
        # For Azure OpenAI, also add:
        # api_version="2024-02-01"
    )

    print("=== Using REST API ===")

    async with llm:
        for i, text in enumerate(texts):
            result = await llm(text)
            print(f"Text {i+1}: {text}")
            print(f"Embedding dimensions: {len(result.output.embeddings[0])}")
            print(f"First 5 values: {result.output.embeddings[0][:5]}")
            print(f"Usage: {result.output.usage}")
            print("---")

    return llm

# Run the REST API test
rest_llm = await test_rest_embeddings()

=== Using REST API ===
[Embedding(embedding=[0.029912294819951057, -0.05337461456656456, -0.011677050031721592, -0.06571181118488312, 0.013148856349289417, 0.061989009380340576, -0.011276631616055965, 7.072414155118167e-05, -0.0599544532597065, 0.014999435283243656, -0.018159490078687668, -0.04592900723218918, 0.03248795494437218, -0.011265809647738934, -0.05194609612226486, 0.031232591718435287, 0.0008245090139098465, 0.06800609827041626, 0.15324099361896515, -0.01028640940785408, -0.040214937180280685, 0.034197848290205, 0.03597267344594002, 0.09592712670564651, -0.014858747832477093, -0.03915436938405037, -0.002783986274152994, 0.07038696110248566, 0.04852631315588951, -0.030085448175668716, 0.009377352893352509, 0.009572151117026806, -0.04382951930165291, 0.03153561055660248, -0.009431463666260242, -0.017131390050053596, 0.0307780634611845, 0.01660110615193844, -0.011190054938197136, 0.023354100063443184, -0.0031952261924743652, -0.019988425076007843, -0.010502851568162441, -0.0003

In [15]:
sdk_result.output.raw_model.usage

Usage(prompt_tokens=6, total_tokens=6)

In [16]:
sdk_result.output.raw_model.object

'list'

In [17]:
sdk_result.output.raw_model.model

'text-embedding-3-large'

## Comparison of Results

Both implementations should produce identical results since they're calling the same OpenAI API endpoints.

In [ ]:
# Compare embeddings from both methods
async def compare_methods():
    """Compare results from both methods."""

    # Test with the same text
    test_text = "This is a test sentence for comparison."

    # SDK result
    config = PublicOpenAIConfig(api_key=API_KEY, base_url=BASE_URL, model=MODEL)
    sdk_llm = create_openai_embeddings_llm(config)
    sdk_result = await sdk_llm(test_text)

    # REST result
    rest_llm = create_openai_embeddings_rest_llm(
        base_url=BASE_URL, api_key=API_KEY, model=MODEL
    )

    async with rest_llm:
        rest_result = await rest_llm(test_text)

    print(f"SDK embedding dimensions: {len(sdk_result.embeddings[0])}")
    print(f"REST embedding dimensions: {len(rest_result.embeddings[0])}")

    print(f"SDK first 10 values: {sdk_result.embeddings[0][:10]}")
    print(f"REST first 10 values: {rest_result.embeddings[0][:10]}")

    # Calculate similarity (should be very close to 1.0)
    import numpy as np

    sdk_vec = np.array(sdk_result.embeddings[0])
    rest_vec = np.array(rest_result.embeddings[0])

    # Cosine similarity
    similarity = np.dot(sdk_vec, rest_vec) / (np.linalg.norm(sdk_vec) * np.linalg.norm(rest_vec))
    print(f"Cosine similarity between SDK and REST results: {similarity:.6f}")

    # Should be very close to 1.0 if both are working correctly
    if similarity > 0.999:
        print("✅ Both methods produce nearly identical results!")
    else:
        print("⚠️ Results differ significantly")

await compare_methods()

## When to Use Each Method

### OpenAI SDK Method (`create_openai_embeddings_llm`)
- **Pros:**
  - Full OpenAI SDK features and error handling
  - Automatic retries and rate limiting built into the SDK
  - Type safety with OpenAI's response objects
  - Better integration with OpenAI's ecosystem

- **Cons:**
  - Requires the OpenAI SDK dependency
  - Less control over HTTP requests
  - May have larger dependency footprint

### REST API Method (`create_openai_embeddings_rest_llm`)
- **Pros:**
  - Direct HTTP control with httpx
  - Smaller dependency footprint (just httpx)
  - More flexibility for custom authentication or headers
  - Works with any OpenAI-compatible API
  - Better for environments where the OpenAI SDK isn't available

- **Cons:**
  - Manual HTTP error handling
  - Need to implement some OpenAI SDK features manually
  - Requires understanding of the REST API structure

### Recommendation
- Use the **SDK method** for most production applications
- Use the **REST API method** when you need more control, have environment constraints, or are working with OpenAI-compatible APIs that may not work perfectly with the official SDK